## Path setup and load everything

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import attacks
import evaluation

import numpy as np
import torch
import joblib

# Load the preprocessed arrays and fitted objects from Stage 2.
data = np.load(config.PROCESSED_DIR / "stage2_arrays.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
label_encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")
class_names = list(label_encoder.classes_)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loaded. X_test: {X_test.shape}  device: {device}")

Loaded. X_test: (718, 9)  device: cuda


## Retrain the baseline CNN

In [2]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights as same as stage 3
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(w, dtype=torch.float32, device=device)

# Train the baseline CNN on the full duplicated training set
cnn = models.CNN1D(n_features=9, n_classes=6)
cnn = models.train_cnn(
    cnn, X_train, y_train,
    n_epochs=50, device=device,
    class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

print("Baseline CNN trained.")

    epoch   1/50     loss 1.6412
    epoch   5/50     loss 0.1999
    epoch  10/50     loss 0.0124
    epoch  15/50     loss 0.0040
    epoch  20/50     loss 0.0027
    epoch  25/50     loss 0.0019
    epoch  30/50     loss 0.0014
    epoch  35/50     loss 0.0011
    epoch  40/50     loss 0.0012
    epoch  45/50     loss 0.0011
    epoch  50/50     loss 0.0011
Baseline CNN trained.


## Wrap CNN and generate FGSM examples

In [3]:
# Wrap the trained CNN
classifier = attacks.wrap_cnn_for_art(cnn, n_features=9, n_classes=6, device=device)

# Baseline: how does the CNN do on the clean test set
cnn.eval()
with torch.no_grad():
    clean_logits = cnn(torch.tensor(X_test, dtype=torch.float32, device=device))
    clean_pred = clean_logits.argmax(dim=1).cpu().numpy()

from sklearn.metrics import f1_score
clean_f1 = f1_score(y_test, clean_pred, average="macro", zero_division=0)
print(f"CNN clean macro-F1 (reference): {clean_f1:.4f}\n")

# Generate FGSM adversarial examples from the test set at a moderate epsilon
X_adv_fgsm = attacks.generate_fgsm(classifier, X_test, epsilon=0.10)

# How does the CNN do on the purturbated test set
with torch.no_grad():
    adv_logits = cnn(torch.tensor(X_adv_fgsm, dtype=torch.float32, device=device))
    adv_pred = adv_logits.argmax(dim=1).cpu().numpy()

adv_f1 = f1_score(y_test, adv_pred, average="macro", zero_division=0)
print(f"CNN FGSM macro-F1 (epsilon=0.10): {adv_f1:.4f}")
print(f"Degredation: {clean_f1 - adv_f1:.4f} drop")

CNN clean macro-F1 (reference): 0.6755

CNN FGSM macro-F1 (epsilon=0.10): 0.1427
Degredation: 0.5328 drop


## Epsilon sweep, FGSM and PGD, against both models

In [4]:
from sklearn.metrics import f1_score

# Helper: get macro-F1 for a model's predictions on some inputs
def cnn_macro_f1(X):
    cnn.eval()
    with torch.no_grad():
        pred = cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(dim=1).cpu().numpy()
    return f1_score(y_test, pred, average="macro", zero_division=0)

def rf_macro_f1(X):
    return f1_score(y_test, rf.predict(X), average="macro", zero_division=0)

# Retrain the RF so the notebook is self-contained
rf = models.build_random_forest(random_seed=config.RANDOM_SEED)
rf.fit(X_train, y_train)

# Clean baselines
clean_cnn = cnn_macro_f1(X_test)
clean_rf = rf_macro_f1(X_test)
print(f"CLEAN   CNN={clean_cnn:.4f}     RF={clean_rf:.4f}\n")

# Sweep over the epsilon list from config for both attacks
epsilons = config.FGSM_EPSILONS

results = {"eps": [], "fgsm_cnn": [], "fgsm_rf": [], "pgd_cnn": [], "pgd_rf": []}

for eps in epsilons:
    # FGSM at this epsilon crafted on the cnn
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # PGD at this epsilon crafted on the cnn
    Xp = attacks.generate_pgd(classifier, X_test, epsilon=eps)

    # Feed same cradfted frames to both models (transfer attack)
    results["eps"].append(eps)
    results["fgsm_cnn"].append(cnn_macro_f1(Xf))
    results["fgsm_rf"].append(rf_macro_f1(Xf))
    results["pgd_cnn"].append(cnn_macro_f1(Xp))
    results["pgd_rf"].append(rf_macro_f1(Xp))

    print(f"eps={eps:.2f} | FGSM: CNN={results['fgsm_cnn'][-1]:.4f} RF={results['fgsm_rf'][-1]:.4f}"
          f" | PGD: CNN={results['pgd_cnn'][-1]:.4f} RF={results['pgd_rf'][-1]:.4f}")

CLEAN   CNN=0.6755     RF=0.7761



PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.01 | FGSM: CNN=0.6755 RF=0.5003 | PGD: CNN=0.6755 RF=0.5003


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.05 | FGSM: CNN=0.3942 RF=0.1465 | PGD: CNN=0.3770 RF=0.1479


...

eps=0.10 | FGSM: CNN=0.1427 RF=0.1466 | PGD: CNN=0.1533 RF=0.1455


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.20 | FGSM: CNN=0.1171 RF=0.1656 | PGD: CNN=0.1186 RF=0.1459


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.30 | FGSM: CNN=0.1514 RF=0.1656 | PGD: CNN=0.1182 RF=0.1589


## Verify the perturbation and break down per-class

In [5]:
from sklearn.metrics import classification_report

# Part 1: is the perturbation real at eps=0.01
X_adv_001 = attacks.generate_fgsm(classifier, X_test, epsilon=0.01)

# How much did the inputs actually change?
diff = np.abs(X_adv_001 - X_test)
print("PERTURBATION CHECK at eps=0.01")
print(f"    max abs change per feature  : {diff.max():.5f}")
print(f"    mean abs change             : {diff.mean():.5f}")
print(f"    rows that changed at all    : {(diff.sum(axis=1) > 0).sum()} / {len(X_test)}")

# Did the CNN predictions actually change?
cnn.eval()
with torch.no_grad():
    p_clean = cnn(torch.tensor(X_test, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    p_adv = cnn(torch.tensor(X_adv_001, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
print(f"    CNN predictions changed     : {(p_clean != p_adv).sum()} / {len(X_test)}")

# Part 2: per-class breakdown, RF under FGSM eps=0.01
print("\nRF per-class CLEAN:")
print(classification_report(y_test, rf.predict(X_test), target_names=class_names, zero_division=0))
print("\nRF per-class under FGSM eps=0.01:")
print(classification_report(y_test, rf.predict(X_adv_001), target_names=class_names, zero_division=0))

PERTURBATION CHECK at eps=0.01
    max abs change per feature  : 0.01000
    mean abs change             : 0.00876
    rows that changed at all    : 718 / 718
    CNN predictions changed     : 0 / 718

RF per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       1.00      0.75      0.86         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       1.00      1.00      1.00         1
spoofing-STEERING_WHEEL       1.00      1.00      1.00         1

               accuracy                           1.00       718
              macro avg       0.78      0.79      0.78       718
           weighted avg       1.00      1.00      1.00       718


RF per-class under FGSM eps=0.01:
                         precision    recall  f1-score   support

                    DoS 

## Integer-valid realism check

In [6]:
# Load the fitted scaler to map [0,1] with real CAN byte space (0-255)
scaler = joblib.load(config.PROCESSED_DIR / "feature_scaler.joblib")

def round_to_valid_can(X_scaled):
    """
    Round adversarial frames to legal integer CAN bytes, then re-scale.

    Steps: inverse transform [0,1] -> original 0-255 space, round to nearest integer, clip to 0-255 and then re-scale back to [0-1] for the models.
    The result is a frame an attacker could actually transmit on the bus.
    """
    X_real = scaler.inverse_transform(X_scaled)
    X_real = np.clip(np.round(X_real), config.FEATURE_MIN, config.FEATURE_MAX)
    X_valid = scaler.transform(X_real)
    return X_valid


print("INTEGER-VALID REALISM CHECK (FGSM)\n")
print(f"{'eps':>6} | {'continuous':>22} | {'integer-valid':>22}")
print(f"{'':>6} | {'CNN':>10} {'RF':>10} | {'CNN':>10} {'RF':>10}")

for eps in config.FGSM_EPSILONS:
    # Continuous attack as before
    Xc = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # Integer-valid version of the same attack
    Xv = round_to_valid_can(Xc)

    row = (cnn_macro_f1(Xc), rf_macro_f1(Xc), cnn_macro_f1(Xv), rf_macro_f1(Xv))
    print(f"{eps:>6.2f} | {row[0]:>10.4f} {row[1]:>10.4f} | {row[2]:>10.4f} {row[3]:>10.4f}")

INTEGER-VALID REALISM CHECK (FGSM)

   eps |             continuous |          integer-valid
       |        CNN         RF |        CNN         RF
  0.01 |     0.6755     0.5003 |     0.3796     0.1486
  0.05 |     0.3942     0.1465 |     0.2930     0.1267
  0.10 |     0.1427     0.1466 |     0.2771     0.1461
  0.20 |     0.1171     0.1656 |     0.1320     0.1440
  0.30 |     0.1514     0.1656 |     0.1334     0.1469


## CNN per-class breakdown under attack

In [7]:
# It is confirmed that the CNN is untouched at eps=0.01.
# the interesting CNN collapse happen at eps=0.05.
# Therefore, let's see which classes fail there, to mirror the RF per-class analysis

X_adv_05 = attacks.generate_fgsm(classifier, X_test, epsilon=0.05)

cnn.eval()
with torch.no_grad():
    cnn_pred_05 = cnn(torch.tensor(X_adv_05, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
print("CNN per-class CLEAN:")
print(classification_report(y_test, clean_pred, target_names=class_names, zero_division=0))
print("\nCNN per-class under FGSM eps=0.05:")
print(classification_report(y_test, cnn_pred_05, target_names=class_names, zero_division=0))

CNN per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       0.80      1.00      0.89         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM       0.50      0.50      0.50         2
         spoofing-SPEED       0.50      1.00      0.67         1
spoofing-STEERING_WHEEL       0.00      0.00      0.00         1

               accuracy                           0.99       718
              macro avg       0.63      0.75      0.68       718
           weighted avg       0.99      0.99      0.99       718


CNN per-class under FGSM eps=0.05:
                         precision    recall  f1-score   support

                    DoS       0.03      0.25      0.06         4
                 benign       1.00      0.76      0.86       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM      

## Robust-support metric and the dual-metric sweep

In [9]:
from sklearn.metrics import f1_score

ROBUST_LABELS = [0, 1, 3]       # DoS, benign, spoofing-RPM

def macro_f1_full(y_true, y_pred):
    """6-class macro-F1"""
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def macro_f1_robust(y_true, y_pred):
    """Macro-F1 over only the robust-support classes"""
    return f1_score(y_true, y_pred, labels=ROBUST_LABELS, average="macro", zero_division=0)

def cnn_pred(X):
    cnn.eval()
    with torch.no_grad():
        return cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
# Regenerate the FGSM sweep reporting both metrics for both models
print("DUAL-METRIC FGSM SWEEP  (full 6-class | robust-support 3-class)\n")
print(f"{'eps':>6} | {'CNN full':>9} {'CNN rob':>8} | {'RF full':>9} {'RF rob':>8}")

clean_cp = cnn_pred(X_test); clean_rp = rf.predict(X_test)
print(f"{'clean':>6} | {macro_f1_full(y_test,clean_cp):>9.4f} {macro_f1_robust(y_test,clean_cp):>8.4f} "
      f"| {macro_f1_full(y_test,clean_rp):>9.4f} {macro_f1_robust(y_test,clean_rp):>8.4f}")

for eps in config.FGSM_EPSILONS:
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    cp = cnn_pred(Xf); rp = rf.predict(Xf)
    print(f"{eps:>6.2f} | {macro_f1_full(y_test,cp):>9.4f} {macro_f1_robust(y_test,cp):>8.4f} "
          f"| {macro_f1_full(y_test,rp):>9.4f} {macro_f1_robust(y_test,rp):>8.4f}")

DUAL-METRIC FGSM SWEEP  (full 6-class | robust-support 3-class)

   eps |  CNN full  CNN rob |   RF full   RF rob
 clean |    0.6755   0.7954 |    0.7761   0.8855
  0.01 |    0.6755   0.7954 |    0.5003   0.6672
  0.05 |    0.3942   0.4551 |    0.1465   0.2929
  0.10 |    0.1427   0.2854 |    0.1466   0.2932
  0.20 |    0.1171   0.2343 |    0.1656   0.3312
  0.30 |    0.1514   0.3028 |    0.1656   0.3312


# Train both defended CNNs

In [10]:
import defense

# Defended model 1: PGD only adversarial training
print(">>> Training defended CNN (PGD-only)")
cnn_pgd = defense.adversarial_train_cnn(
    X_train, y_train, strategy="pgd",
    epsilons=config.FGSM_EPSILONS, n_epochs=50,
    device=device, class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

# Defended model 2: multi-strategy (FGSM+PGD)
print("\n>>> Training defended CNN (multi-strategy)")
cnn_multi = defense.adversarial_train_cnn(
    X_train, y_train, strategy="multi",
    epsilons=config.FGSM_EPSILONS, n_epochs=50,
    device=device, class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

print("\nBoth defended models trained.")

>>> Training defended CNN (PGD-only)
    epoch   1/50     loss 1.5818
    epoch   5/50     loss 0.1103
    epoch  10/50     loss 0.0083
    epoch  15/50     loss 0.0026
    epoch  20/50     loss 0.0019
    epoch  25/50     loss 0.0013
    epoch  30/50     loss 0.0010
    epoch  35/50     loss 0.0008
    epoch  40/50     loss 0.0009
    epoch  45/50     loss 0.0006
    epoch  50/50     loss 0.0007


PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

    augmented trainset: 3838 clean -> 42218 total (multi strategy)
    epoch   1/50     loss 0.5532
    epoch   5/50     loss 0.0028
    epoch  10/50     loss 0.0009
    epoch  15/50     loss 0.0043
    epoch  20/50     loss 0.0005
    epoch  25/50     loss 0.0005
    epoch  30/50     loss 0.0001
    epoch  35/50     loss 0.0002
    epoch  40/50     loss 0.0000
    epoch  45/50     loss 0.0007
    epoch  50/50     loss 0.0000

Both defended models trained.


## Full evaluation of all four models across the attack sweep

In [11]:
# Predict helper for any CNN model
def pred_of(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
# The models to compare
def eval_all_on(X):
    return {
        "base_cnn":     macro_f1_robust(y_test, pred_of(cnn, X)),
        "def_pgd":      macro_f1_robust(y_test, pred_of(cnn_pgd, X)),
        "def_multi":    macro_f1_robust(y_test, pred_of(cnn_multi, X)),
        "rf":           macro_f1_robust(y_test, rf.predict(X)),
    }

print("ROBUST-SUPPORT macro-F1 (benign, DoS, RPM) under PGD attack\n")
print(f"{'eps':>6} | {'base':>7} {'def_pgd':>7} {'def_multi':>9} {'rf':>7}")

# Clean row
c = eval_all_on(X_test)
print(f"{'clean':>6} | {c['base_cnn']:>7.4f} {c['def_pgd']:>7.4f} {c['def_multi']:>9.4f} {c['rf']:>7.4f}")

# PGD attack sweep (The defence's real test as PGD is the stronger attack.)
for eps in config.FGSM_EPSILONS:
    Xp = attacks.generate_pgd(classifier, X_test, epsilon=eps)
    r = eval_all_on(Xp)
    print(f"{eps:>6.2f} | {r['base_cnn']:>7.4f} {r['def_pgd']:>7.4f} {r['def_multi']:>9.4f} {r['rf']:>7.4f}")

ROBUST-SUPPORT macro-F1 (benign, DoS, RPM) under PGD attack

   eps |    base def_pgd def_multi      rf
 clean |  0.7954  0.8514    0.8884  0.8855


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.01 |  0.7954  0.8514    0.8884  0.6672


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.05 |  0.4207  0.8822    0.8408  0.2958


...

  0.10 |  0.3065  0.7420    0.8882  0.2909


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.20 |  0.2371  0.5144    0.9174  0.2918


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.30 |  0.2363  0.4808    0.8070  0.3178


## Defense tested against FGSM attacks

In [12]:
print("ROBUST-SUPPORT macro-F1 under FGSM attack (defence generalisation check)\n")
print(f"{'eps':>6} | {'base':>7} {'def_pgd':>7} {'def_multi':>9} {'rf':>7}")

# Clean row
c = eval_all_on(X_test)
print(f"{'clean':>6} | {c['base_cnn']:>7.4f} {c['def_pgd']:>7.4f} {c['def_multi']:>9.4f} {c['rf']:>7.4f}")

# FGSM attack sweep this time (the test attacks are FGSM, not PGD)
for eps in config.FGSM_EPSILONS:
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    r = eval_all_on(Xf)
    print(f"{eps:>6.2f} | {r['base_cnn']:>7.4f} {r['def_pgd']:>7.4f} {r['def_multi']:>9.4f} {r['rf']:>7.4f}")

ROBUST-SUPPORT macro-F1 under FGSM attack (defence generalisation check)

   eps |    base def_pgd def_multi      rf
 clean |  0.7954  0.8514    0.8884  0.8855
  0.01 |  0.7954  0.8514    0.8884  0.6672
  0.05 |  0.4551  0.8822    0.8884  0.2929
  0.10 |  0.2854  0.6531    0.7766  0.2932
  0.20 |  0.2343  0.2864    0.8192  0.3312
  0.30 |  0.3028  0.2867    0.7389  0.3312


## Defended CV

In [13]:
import numpy as np
import pandas as pd
import crossval

strict = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_strict.csv")

print("DEFENDED-MODEL CROSS-VALIDATION (PGD @ eps=0.10, 2-fold)\n")
cv_def = crossval.crossval_defended(
    strict,
    config.FEATURE_COLUMNS,
    attack_eps=0.10,
    n_splits=2,
    device=device,
    random_seed=config.RANDOM_SEED,
    cnn_epochs=50,
)

def ms(key):
    v = cv_def[key]
    return f"{np.mean(v):.4f} ± {np.std(v):.4f}"

print("\n" + "="*60)
print("ROBUST-SUPPORT macro-F1 (mean ± fold-spread)")
print("="*60)
print(f"{'model':>12} | {'clean':>16} | {'PGD eps=0.10':>16}")
print(f"{'base CNN':>12} | {ms('base_clean'):>16} | {ms('base_adv'):>16}")
print(f"{'def PGD':>12} | {ms('pgd_clean'):>16} | {ms('pgd_adv'):>16}")
print(f"{'def multi':>12} | {ms('multi_clean'):>16} | {ms('multi_adv'):>16}")
print(f"{'RF':>12} | {ms('rf_clean'):>16} | {ms('rf_adv'):>16}")

DEFENDED-MODEL CROSS-VALIDATION (PGD @ eps=0.10, 2-fold)

Label mapping:
    0 -> DoS
    1 -> benign
    2 -> spoofing-GAS
    3 -> spoofing-RPM
    4 -> spoofing-SPEED
    5 -> spoofing-STEERING_WHEEL
    epoch   1/50     loss 1.7333
    epoch   5/50     loss 0.2429
    epoch  10/50     loss 0.0142
    epoch  15/50     loss 0.0047
    epoch  20/50     loss 0.0029
    epoch  25/50     loss 0.0019
    epoch  30/50     loss 0.0015
    epoch  35/50     loss 0.0009
    epoch  40/50     loss 0.0008
    epoch  45/50     loss 0.0010
    epoch  50/50     loss 0.0008
    epoch   1/50     loss 1.7068
    epoch   5/50     loss 0.2200
    epoch  10/50     loss 0.0102
    epoch  15/50     loss 0.0032
    epoch  20/50     loss 0.0018
    epoch  25/50     loss 0.0012
    epoch  30/50     loss 0.0009
    epoch  35/50     loss 0.0005
    epoch  40/50     loss 0.0004
    epoch  45/50     loss 0.0005
    epoch  50/50     loss 0.0003


PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

    augmented trainset: 2774 clean -> 30514 total (multi strategy)
    epoch   1/50     loss 0.5853
    epoch   5/50     loss 0.0054
    epoch  10/50     loss 0.0025
    epoch  15/50     loss 0.0017
    epoch  20/50     loss 0.0006
    epoch  25/50     loss 0.0003
    epoch  30/50     loss 0.0004
    epoch  35/50     loss 0.0003
    epoch  40/50     loss 0.0002
    epoch  45/50     loss 0.0002
    epoch  50/50     loss 0.0004


PGD - Batches:   0%|          | 0/57 [00:00<?, ?it/s]

  fold 2: robust labels = [0, 1, 3] (names=['DoS', 'benign', 'spoofing-GAS', 'spoofing-RPM', 'spoofing-SPEED', 'spoofing-STEERING_WHEEL'])
  fold 2: base_adv=0.3217 pgd_adv=0.7097 multi_adv=0.6256 rf_adv=0.2832

ROBUST-SUPPORT macro-F1 (mean ± fold-spread)
       model |            clean |     PGD eps=0.10
    base CNN |  0.6460 ± 0.0201 |  0.3046 ± 0.0171
     def PGD |  0.7463 ± 0.0947 |  0.6871 ± 0.0226
   def multi |  0.6860 ± 0.0634 |  0.6059 ± 0.0197
          RF |  0.8528 ± 0.0039 |  0.2975 ± 0.0143


## Inference latency benchmark (CPU, single-frame)

In [14]:
import time
import numpy as np

# Move the CNN to CPU for an edge-realistic measurement
cnn_cpu = cnn.to("cpu")
cnn_cpu.eval()

# A single test frame, shaed as one sample
single = X_test[:1].astype(np.float32)

# --- CNN single-frame latency ---
import torch
single_t = torch.tensor(single, dtype=torch.float32)
# Warm up
with torch.no_grad():
    for _ in range(10):
        _ = cnn_cpu(single_t)

# Times runs
n_runs = 1000
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_runs):
        _ = cnn_cpu(single_t)
cnn_ms = (time.perf_counter() - t0) / n_runs * 1000

# --- RF single-frame latency ---
for _ in range(10):
    _ = rf.predict(single)
t0 = time.perf_counter()
for _ in range(n_runs):
    _ = rf.predict(single)
rf_ms = (time.perf_counter() - t0) / n_runs * 1000

print("INFERENCE LATENCY (CPU, single frame, mean of 1000 runs)")
print(f"  RF  : {rf_ms:.4f} ms/frame")
print(f"  CNN : {cnn_ms:.4f} ms/frame  (defended CNN identical - same architecture)")
print(f"\n  CAN frames can arrive ~1-2 ms apart on a busy bus; compare against that.")

# Restore CNN to original device for any further GPU work
cnn = cnn_cpu.to(device)

INFERENCE LATENCY (CPU, single frame, mean of 1000 runs)
  RF  : 6.8291 ms/frame
  CNN : 0.0553 ms/frame  (defended CNN identical - same architecture)

  CAN frames can arrive ~1-2 ms apart on a busy bus; compare against that.


## Investigate the def_pgd-above-clean oddity

In [26]:
from sklearn.metrics import f1_score, confusion_matrix

# Re-evaluate def_pgd on clean and on FGSM eps=0.01
def robust_f1_and_preds(model, X):
    model.eval()
    with torch.no_grad():
        p = model(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    return f1_score(y_test, p, labels=[0,1,3], average="macro", zero_division=0), p

# clean
f1_clean, pred_clean = robust_f1_and_preds(cnn_pgd, X_test)
# Mild attack
X_mild = attacks.generate_fgsm(classifier, X_test, epsilon=0.01)
f1_mild, pred_mild = robust_f1_and_preds(cnn_pgd, X_mild)

print(f"def_pgd robust-support F1  clean={f1_clean:.4f}  FGSM@0.01={f1_mild:.4f}")
print(f"predictions that CHANGED clean->mild: {(pred_clean != pred_mild).sum()} / {len(y_test)}\n")

# For the robust support classes show how many each got right under each condition
for lbl, name in [(0,"DoS"), (1,"benign"), (3,"RPM")]:
    mask = (y_test == lbl)
    correct_clean = (pred_clean[mask] == lbl).sum()
    correct_mild = (pred_mild[mask] == lbl).sum()
    print(f"  {name:7s} (n={mask.sum():3d}): correct clean={correct_clean}, correct mild={correct_mild}")

def_pgd robust-support F1  clean=0.8514  FGSM@0.01=0.8514
predictions that CHANGED clean->mild: 0 / 718

  DoS     (n=  4): correct clean=4, correct mild=4
  benign  (n=709): correct clean=708, correct mild=708
  RPM     (n=  2): correct clean=1, correct mild=1


## Adversarial-set diversity and leakage diagnostic

In [ ]:
import numpy as np

# Rebuild the PGD-strategy adversarial training set from the SAME base classifier
# used earlier (the baseline CNN's wrapper, `classifier`).
X_aug, y_aug = defense.build_adversarial_trainset(
    classifier, X_train, y_train, strategy="pgd", epsilons=config.FGSM_EPSILONS
)

# --- Part 1: distinct adversarial signatures per class (the diversity question) ---
def distinct_per_class(X, y, decimals=4):
    Xr = np.round(X, decimals)
    report = {}
    for cls in np.unique(y):
        rows = Xr[y == cls]
        n_total = len(rows)
        n_unique = len(np.unique(rows, axis=0))
        report[int(cls)] = (n_total, n_unique)
    return report

print("ADVERSARIAL TRAIN SET — distinct signatures per class")
print(f"{'class':>6} | {'total rows':>10} | {'unique':>7}")
rep = distinct_per_class(X_aug, y_aug)
for cls, (tot, uniq) in sorted(rep.items()):
    print(f"{cls:>6} | {tot:>10} | {uniq:>7}")

# --- Part 2: leakage check vs the TEST adversarial set ---
# Craft the test-side PGD set (from TEST signatures) and check for ANY overlap.
X_test_adv = attacks.generate_pgd(classifier, X_test, epsilon=0.10)

def row_set(X, decimals=4):
    return set(map(tuple, np.round(X, decimals)))

train_adv_keys = row_set(X_aug)
test_adv_keys = row_set(X_test_adv)
overlap = train_adv_keys & test_adv_keys

print(f"\nLEAKAGE CHECK (train-adv vs test-adv)")
print(f"  unique train-adv rows: {len(train_adv_keys)}")
print(f"  unique test-adv rows : {len(test_adv_keys)}")
print(f"  overlap (must be 0)  : {len(overlap)}")

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

ADVERSARIAL TRAIN SET — distinct signatures per class
 class | total rows |  unique
     0 |       1200 |     102
     1 |      17028 |   12448
     2 |       1200 |       6
     3 |       1200 |      48
     4 |       1200 |      24
     5 |       1200 |      12


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]


LEAKAGE CHECK (train-adv vs test-adv)
  unique train-adv rows: 12640
  unique test-adv rows : 580
  overlap (must be 0)  : 209


In [28]:
import numpy as np

def keyed_rows(X, y, decimals=4):
    """Map each rounded feature-tuple to the set of class labels it appears with."""
    Xr = np.round(X, decimals)
    d = {}
    for row, lbl in zip(map(tuple, Xr), y):
        d.setdefault(row, set()).add(int(lbl))
    return d

# Train-adversarial keyed by label.
train_keyed = keyed_rows(X_aug, y_aug)

# Test-adversarial: we need its labels too. y_test aligns with X_test, and the
# PGD attack preserves row order, so y_test labels the crafted test rows.
test_keyed = keyed_rows(X_test_adv, y_test)

# Find the overlapping feature-tuples and see which classes they belong to.
overlap_keys = set(train_keyed) & set(test_keyed)
print(f"Total overlapping rows: {len(overlap_keys)}\n")

# For each overlap, record (train labels, test labels).
from collections import Counter
test_class_counter = Counter()
same_class = 0
diff_class = 0
for k in overlap_keys:
    tr_labels = train_keyed[k]
    te_labels = test_keyed[k]
    for tl in te_labels:
        test_class_counter[tl] += 1
    # Does the test row's class also appear in train for this same tuple?
    if te_labels & tr_labels:
        same_class += 1
    else:
        diff_class += 1

names = {0:"DoS",1:"benign",2:"GAS",3:"RPM",4:"SPEED",5:"STEERING"}
print("Overlapping rows by TEST class:")
for cls, cnt in sorted(test_class_counter.items()):
    print(f"  {names[cls]:9s}: {cnt}")

print(f"\nOverlaps where train and test share the class (true leakage): {same_class}")
print(f"Overlaps where classes differ (collision, less harmful)    : {diff_class}")

Total overlapping rows: 209

Overlapping rows by TEST class:
  benign   : 209

Overlaps where train and test share the class (true leakage): 209
Overlaps where classes differ (collision, less harmful)    : 0


In [29]:
# Leakage check WITHOUT rounding - compare raw float rows exactly.
def row_set_exact(X):
    # Each row becomes a tuple of its exact float values. No rounding.
    return set(map(tuple, X))

train_adv_exact = row_set_exact(X_aug)
test_adv_exact = row_set_exact(X_test_adv)
overlap_exact = train_adv_exact & test_adv_exact

print("LEAKAGE CHECK (no rounding, exact float match)")
print(f"  unique train-adv rows: {len(train_adv_exact)}")
print(f"  unique test-adv rows : {len(test_adv_exact)}")
print(f"  overlap (exact)      : {len(overlap_exact)}")

LEAKAGE CHECK (no rounding, exact float match)
  unique train-adv rows: 12780
  unique test-adv rows : 597
  overlap (exact)      : 221


In [30]:
from sklearn.metrics import f1_score, accuracy_score

# Use the PGD-defended model (the headline defended model).
defended = cnn_pgd

def split_report(model, X, y, label):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    correct = (pred == y)
    rob = f1_score(y, pred, labels=[0,1,3], average="macro", zero_division=0)
    full = f1_score(y, pred, average="macro", zero_division=0)
    print(f"\n{label}")
    print(f"  overall accuracy : {accuracy_score(y, pred):.4f}  ({correct.sum()}/{len(y)} correct)")
    print(f"  macro-F1 (6-class): {full:.4f}   robust-support F1: {rob:.4f}")
    # per-class right/wrong
    names = {0:"DoS",1:"benign",2:"GAS",3:"RPM",4:"SPEED",5:"STEERING"}
    for c in np.unique(y):
        m = (y == c)
        print(f"    {names[int(c)]:9s}: {correct[m].sum()}/{m.sum()} correct")

# Clean test data
split_report(defended, X_test, y_test, "DEFENDED model on CLEAN test data")

# Adversarial test data (PGD @ 0.10, crafted from baseline CNN - the transfer threat)
X_test_adv_010 = attacks.generate_pgd(classifier, X_test, epsilon=0.10)
split_report(defended, X_test_adv_010, y_test, "DEFENDED model on ADVERSARIAL test data (PGD 0.10)")


DEFENDED model on CLEAN test data
  overall accuracy : 0.9958  (715/718 correct)
  macro-F1 (6-class): 0.7035   robust-support F1: 0.8514
    DoS      : 4/4 correct
    benign   : 708/709 correct
    GAS      : 1/1 correct
    RPM      : 1/2 correct
    SPEED    : 1/1 correct
    STEERING : 0/1 correct


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]


DEFENDED model on ADVERSARIAL test data (PGD 0.10)
  overall accuracy : 0.9930  (713/718 correct)
  macro-F1 (6-class): 0.5376   robust-support F1: 0.7420
    DoS      : 4/4 correct
    benign   : 707/709 correct
    GAS      : 1/1 correct
    RPM      : 1/2 correct
    SPEED    : 0/1 correct
    STEERING : 0/1 correct


## Builda no-duplication training set

In [31]:
import pandas as pd

# Load the strict signatures and re-split exactly as Stage 1 did, but DON'T duplicate.
strict = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_strict.csv")
test_df = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_test.csv")

# The train signatures are the strict rows NOT in the test set.
# Identify test signatures by their feature tuples.
feat = config.FEATURE_COLUMNS
test_keys = set(map(tuple, test_df[feat].values))
train_mask = ~strict[feat].apply(lambda r: tuple(r) in test_keys, axis=1)
train_sig_nodup = strict[train_mask].reset_index(drop=True)

print("Un-duplicated train signatures per class:")
print(train_sig_nodup["true_class"].value_counts())

# Encode + scale using the SAME fitted encoder/scaler (no leakage - they were fit on train).
import joblib
scaler = joblib.load(config.PROCESSED_DIR / "feature_scaler.joblib")
encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")

X_train_nodup = scaler.transform(train_sig_nodup[feat].values).astype(np.float32)
y_train_nodup = encoder.transform(train_sig_nodup["true_class"].values)

print(f"\nNo-dup train set: {X_train_nodup.shape} (vs duplicated {X_train.shape})")

Un-duplicated train signatures per class:
true_class
benign                     2838
DoS                          17
spoofing-RPM                  8
spoofing-SPEED                4
spoofing-STEERING_WHEEL       2
spoofing-GAS                  1
Name: count, dtype: int64

No-dup train set: (2870, 9) (vs duplicated (3838, 9))


## Adversarial Training with no duplication

In [32]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights from the un-duplicated labels (the real imbalance, now extreme).
w_nodup = compute_class_weight("balanced", classes=np.unique(y_train_nodup), y=y_train_nodup)
cw_nodup = torch.tensor(w_nodup, dtype=torch.float32, device=device)
print("Class weights (no-dup):", dict(zip(np.unique(y_train_nodup), np.round(w_nodup, 2))))

# Adversarial-train on the un-duplicated signatures (PGD strategy).
print("\n>>> Training defended CNN on NO-DUPLICATION set (PGD strategy)")
cnn_nodup = defense.adversarial_train_cnn(
    X_train_nodup, y_train_nodup, strategy="pgd",
    epsilons=config.FGSM_EPSILONS, n_epochs=50,
    device=device, class_weights=cw_nodup,
    random_seed=config.RANDOM_SEED,
)

# Evaluate it the same way as the headline model: robust-support under PGD @ 0.10.
X_test_adv_010 = attacks.generate_pgd(classifier, X_test, epsilon=0.10)

print("\nNO-DUPLICATION defended model:")
print(f"  clean robust-support F1      : {macro_f1_robust(y_test, pred_of(cnn_nodup, X_test)):.4f}")
print(f"  PGD@0.10 robust-support F1   : {macro_f1_robust(y_test, pred_of(cnn_nodup, X_test_adv_010)):.4f}")

# Per-class breakdown on clean, to see if the rare classes were learned at all.
from sklearn.metrics import classification_report
print("\nPer-class on CLEAN test (did rare classes survive without duplication?):")
print(classification_report(y_test, pred_of(cnn_nodup, X_test),
      target_names=class_names, zero_division=0))

Class weights (no-dup): {np.int64(0): np.float64(28.14), np.int64(1): np.float64(0.17), np.int64(2): np.float64(478.33), np.int64(3): np.float64(59.79), np.int64(4): np.float64(119.58), np.int64(5): np.float64(239.17)}

>>> Training defended CNN on NO-DUPLICATION set (PGD strategy)
    epoch   1/50     loss 1.4912
    epoch   5/50     loss 0.9728
    epoch  10/50     loss 0.8781
    epoch  15/50     loss 0.5861
    epoch  20/50     loss 0.4428
    epoch  25/50     loss 0.3606
    epoch  30/50     loss 0.1655
    epoch  35/50     loss 0.1500
    epoch  40/50     loss 0.0969
    epoch  45/50     loss 0.0502
    epoch  50/50     loss 0.0576


PGD - Batches:   0%|          | 0/90 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/90 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/90 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/90 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/90 [00:00<?, ?it/s]

    augmented trainset: 2870 clean -> 17220 total (pgd strategy)
    epoch   1/50     loss 1.2141
    epoch   5/50     loss 0.5451
    epoch  10/50     loss 0.1253
    epoch  15/50     loss 0.0683
    epoch  20/50     loss 0.0490
    epoch  25/50     loss 0.0516
    epoch  30/50     loss 0.0241
    epoch  35/50     loss 0.0251
    epoch  40/50     loss 0.0167
    epoch  45/50     loss 0.0101
    epoch  50/50     loss 0.0105


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]


NO-DUPLICATION defended model:
  clean robust-support F1      : 0.7951
  PGD@0.10 robust-support F1   : 0.6142

Per-class on CLEAN test (did rare classes survive without duplication?):
                         precision    recall  f1-score   support

                    DoS       0.80      1.00      0.89         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.50      0.50      0.50         2
         spoofing-SPEED       0.50      1.00      0.67         1
spoofing-STEERING_WHEEL       0.00      0.00      0.00         1

               accuracy                           0.99       718
              macro avg       0.47      0.58      0.51       718
           weighted avg       0.99      0.99      0.99       718

